Goal: To build a binary text classifier that identifies subjective media bias. 

Objective: Upgrade from a sparse Bag-of-Words baseline to a dense sequence-aware model. 
This notebook implements a PyTorch RNN using a ReLU activation function within the recurrent layer to prevent vanishing gradients, concluding with a Softmax output layer for binary classification.

Dataset:  newsmediabias/debiased_dataset (streamed from HuggingFace)

In [49]:
# Setup for matrix multiplication

import os
import re
from collections import Counter
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Engine active on device: {device}")

Engine active on device: mps


In [ ]:
# Same data prep as baseline, identical split sizes

dataset = load_dataset("newsmediabias/debiased_dataset", split="train")
df_original = dataset.to_pandas()

# Organize data blocks into distinct class arrays
df_biased = pd.DataFrame({'text': df_original['biased_text'], 'label': 1})
df_unbiased = pd.DataFrame({'text': df_original['debiased_text'], 'label': 0})

df = pd.concat([df_biased, df_unbiased]).sample(frac=1, random_state=42).reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], 
    df['label'], 
    train_size=0.8,
    test_size=0.2,
    random_state=42
)

print(f"Train split size: {len(X_train)} rows") 
print(f"Test split size: {len(X_test)} rows")

Using the latest cached version of the dataset since newsmediabias/debiased_dataset couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/sneharoy/.cache/huggingface/datasets/newsmediabias___debiased_dataset/default/0.0.0/bb97995835c1c645d1d0ffef39bcb7101b3aef9b (last modified on Thu Mar 12 08:18:45 2026).


Train split size: 11987 rows
Test split size: 2997 rows


In [ ]:
# Tokenize (tokens → integer IDs using vocab)

import sys
sys.path.append("../src")
import contractions

def clean_and_tokenize(text):
     for word, expanded in contractions.CONTRACTION_MAP.items():
          text = re.sub(r'\b' + re.escape(word) + r'\b', expanded, text, flags=re.IGNORECASE)
          text = str(text).lower()
          text = re.sub(r'[^\w\s]', '', text)
          text = re.sub(r"\s+n't\b", "n't", text)
          text = re.sub(r"\s+'(s|ve|ll|d|re|m)\b", r"'\1", text)
     return text.split()

# Give every unique word a unique index in dictionary
all_words = []
for text in X_train:
    all_words.extend(clean_and_tokenize(text))

word_counts = Counter(all_words)
print(word_counts)

vocab = {}
for i, (word, count) in enumerate(word_counts.items()):
    vocab[word] = i + 2

vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

print(f"There are {len(vocab)} unique structural tokens.")
print(vocab)
print(len(vocab))

Counter({'the': 28915, 'to': 18267, 'and': 14218, 'of': 13070, 'a': 12238, 'is': 10640, 'in': 9002, 'that': 8751, 'i': 8260, 'it': 8049, 'for': 5950, 'you': 5637, 'not': 4581, 'be': 4402, 'have': 4353, 'are': 4317, 'as': 4158, 'this': 4134, 'on': 4093, 'with': 3863, 'they': 2915, 'was': 2707, 'or': 2567, 's': 2427, 'their': 2426, 'but': 2362, 'by': 2355, 'from': 2276, 'who': 2213, 'if': 2106, 'your': 2057, 'my': 2049, 'do': 2048, 'he': 1977, 'we': 1953, 'about': 1934, 'has': 1921, 'an': 1873, 'would': 1842, 'nt': 1841, 'can': 1829, 'there': 1815, 'at': 1735, 'his': 1705, 'will': 1699, 'all': 1613, 'more': 1550, 'what': 1523, 'its': 1464, 'like': 1458, 'so': 1444, 'no': 1425, 'people': 1424, 'should': 1294, 'our': 1264, 'some': 1244, 'one': 1236, 'when': 1234, 'been': 1232, 'may': 1173, 'just': 1164, 'any': 1139, 'trump': 1137, 'were': 1088, 'which': 1052, 'them': 1039, 'me': 1025, 'individuals': 1018, 'than': 1003, 'these': 1000, 'how': 947, 'out': 935, 'us': 932, 'other': 903, 'up': 9

Instead of capping max_features, we keep all the words (unigram) for the embedding layer so later it learns a dense vector per word.

Each word get an integerID so exts become sequences of integers. Word order is preserved. 

RNN input is a 1D array 11987 × max_count(longest_df) of integer ID each integer ID gets mapped to a learned embedding vector (floats) via nn.Embedding





In [59]:
# Text to tensors

class SequentialBiasDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=64):
        self.texts = texts.reset_index(drop=True).tolist()
        self.labels = labels.reset_index(drop=True).tolist()
        self.vocab = vocab
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        tokens = clean_and_tokenize(self.texts[idx])
        
        # Convert string tokens to integer indices from our vocabulary map
        numerical_sequence = [self.vocab.get(token, self.vocab['<UNK>']) for token in tokens]
        
        # Enforce sequence uniformity: cutoff or pad with zeros up to max_len
        if len(numerical_sequence) > self.max_len:
            numerical_sequence = numerical_sequence[:self.max_len]
        else:
            numerical_sequence = numerical_sequence + [self.vocab['<PAD>']] * (self.max_len - len(numerical_sequence))
            
        return {
            'input_ids': torch.tensor(numerical_sequence, dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

#  1D tensor of 64 integers
train_dataset = SequentialBiasDataset(X_train, y_train, vocab, max_len=64)
test_dataset = SequentialBiasDataset(X_test, y_test, vocab, max_len=64)

# Shuffle batch for every epoch for the training 2D tensor (32, 64) per batch
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) 
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

sample = train_dataset[0]
print(type(sample))
print(sample)

<class 'dict'>
{'input_ids': tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
        20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 18, 30, 31, 21, 32, 33,  9, 34,
        35, 36, 37, 38, 39, 13, 34, 40, 41, 42, 43, 44, 38, 45, 28, 46, 47, 48,
        49, 50, 47, 51, 38, 52, 28, 38, 53, 45]), 'label': tensor(1)}


Most sentences fit within 64 tokens. I didn't do the max length of tokens for any sentence as too much of them can cause the model to struggle to learn from very long sequences anyway due to vanishing gradient.

Each sample in train_dataset is a dictionary containing input_ids (a 64-length tensor of token indices representing the tokenized sentence) and `abel (aka the class: 1 = biased, 0 = neutral).

 The model will learn to predict this label from the token sequence.

In [ ]:
#GloVe embedding

import urllib.request
import zipfile

# 1. Download a lightweight pre-trained word vector file (GloVe 100-dimensional tokens)
glove_url = "https://nlp.stanford.edu/data/glove.6B.zip"
glove_zip_path = "../data/glove.6B.zip"
glove_txt_path = "../data/glove.6B.100d.txt"

os.makedirs("../data", exist_ok=True)

if not os.path.exists(glove_txt_path):
    print("Downloading pre-trained vector space file (This may take a minute)...")
    urllib.request.urlretrieve(glove_url, glove_zip_path)
    print("Extracting vectors...")
    with zipfile.ZipFile(glove_zip_path, 'r') as zip_ref:
        zip_ref.extract("glove.6B.100d.txt", "../data")
    print("Extraction complete!")

# 2. Build the pre-trained weight matrix matching your exact vocab size
embedding_dim = 100
weights_matrix = torch.zeros((len(vocab), embedding_dim))

# Initialize the matrix with random normal values so unknown words have a baseline variance
nn.init.normal_(weights_matrix, mean=0, std=0.6)
weights_matrix[0] = torch.zeros(embedding_dim) # Hardcode <PAD> (Index 0) to all zeros

print("Mapping pre-trained semantic vectors to our vocabulary coordinates...")
matched_count = 0
with open(glove_txt_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        word = parts[0]
        if word in vocab:
            idx = vocab[word]
            weights_matrix[idx] = torch.tensor([float(x) for x in parts[1:]])
            matched_count += 1

print(f"Success! Matched {matched_count} out of {len(vocab)} words with pre-trained semantic vectors.")
# Convert the weight matrix into a permanent PyTorch FloatTensor
weights_matrix = torch.FloatTensor(weights_matrix)
print(weights_matrix)

Mapping pre-trained semantic vectors to our vocabulary coordinates...
Success! Matched 23516 out of 28602 words with pre-trained semantic vectors.
tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.8211, -0.4027, -0.7426,  ..., -0.8490,  0.5004, -0.1526],
        [-0.2263, -0.0592,  0.1805,  ..., -0.7326,  0.0662, -0.7187],
        ...,
        [ 0.0489,  0.3922,  0.4341,  ..., -0.5759,  0.0787,  0.0402],
        [-0.2553, -0.5111,  0.3669,  ...,  0.9538, -1.2734, -0.8362],
        [-0.5572,  0.7320, -0.1954,  ...,  0.0527,  0.7030, -0.0454]])


After building a vocab map of 28,602 unique IDs. weights_matrix is a 2D tensor of shape (28602, 100) matrix to serve as a lookup table. 

When the RNN sees token ID i, it looks up row i in weights_matrix to get vocab's vector in weights_matrix. 

Say the token sequence is [2, 47, 153] → embedding lookup → three 100-dim vectors → fed into RNN

In [69]:
# RNN classifier: Forward pass

class PretrainedReLURNNClassifier(nn.Module):
    def __init__(self, embedding_weights, hidden_dim=64):
        super(PretrainedReLURNNClassifier, self).__init__()
        
        self.embedding = nn.Embedding.from_pretrained(
            embedding_weights, 
            freeze=True, 
            padding_idx=0
        )
        
        self.rnn = nn.RNN(
            input_size=embedding_weights.shape[1], 
            hidden_size=hidden_dim, 
            num_layers=1, 
            nonlinearity='relu', 
            batch_first=True,
            bidirectional=True  
        )
        
        self.dropout = nn.Dropout(p=0.4)
        self.fc = nn.Linear(hidden_dim * 2, 2) 
        
    def forward(self, x):
        # Shape: [batch_size, sequence_length, embedding_dim]
        embedded = self.embedding(x) 
        
        # h_t = ReLU(W_x · x_t + W_h · h_{t-1} + b)
        out, h_n = self.rnn(embedded) 
        
        # Count how many tokens in each row are NOT padding zeros
        lengths = (x != 0).sum(dim=1)
        lengths = torch.clamp(lengths, min=1) # Prevents an error if a line is blank
        
        # Extract the hidden state from 'out' at the last valid token index
        batch_size = x.size(0)
        final_state = out[torch.arange(batch_size), lengths - 1]
        
        x = self.dropout(final_state)

        # z = wx + b
        logits = self.fc(x)
        return logits

# Re-initialize the model with the updated forward propagation tracking loop
model = PretrainedReLURNNClassifier(embedding_weights=weights_matrix, hidden_dim=64).to(device)
print(model)

PretrainedReLURNNClassifier(
  (embedding): Embedding(28602, 100, padding_idx=0)
  (rnn): RNN(100, 64, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


Each batch of token IDs  [32, 64] is converted into 100-dim GloVe vectors via the embedding layer, then through the RNN, which sequentially builds a 64-dim hidden state capturing context from each word. 

The dynamic length fix extracts the hidden state at the last real (non-padding) token of each sentence, which gets passed through dropout (40%) and the output linear layer to create 2 raw scores for the "neutral" and "biased" classes (need softMax).

In [70]:
# Training loop for one epoch

def train_epoch(model, data_loader, optimizer, loss_fn, device):
    model.train() # Activate dropout functions
    total_loss = 0
    correct_predictions = 0
    
    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)
        
        # Clear old gradients
        optimizer.zero_grad()
        
        # Forward pass calculations (z = wx+b)
        logits = model(input_ids)
        
        # Calculate cross-entropy (softmaxed the logits) classification losses
        loss = loss_fn(logits, labels)
        total_loss += loss.item()
        
        # picks the class with the higher logit; count correct predictions
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels).item()
        
        # backpropagation: compute gradients
        loss.backward()
        
        # update weights using gradients
        optimizer.step()
        
    return total_loss / len(data_loader), correct_predictions / len(data_loader.dataset)

This algorithm (forward and backward pass) continues for each patch.

It sums up the loss values across batches, returns per-epoch averages. 

In [72]:
# Performance: no backprop, no weight updates

def eval_model(model, data_loader, loss_fn, device):
    model.eval() # Deactivate dropout scaling arrays
    total_loss = 0
    correct_predictions = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids)
            loss = loss_fn(logits, labels)
            total_loss += loss.item()
            
            _, preds = torch.max(logits, dim=1)
            correct_predictions += torch.sum(preds == labels).item()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    return total_loss / len(data_loader), correct_predictions / len(data_loader.dataset), all_preds, all_labels

At this point, the model has finished learning and starts making predictions, so no backpropagation or weight updates

The model runs on test data.

model.eval() disables dropout for deterministic predictions

torch.no_grad() skips gradient tracking to save memory and speed up computation

In [73]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 5

print("Initiating sequential RNN training processing loop execution...")
for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch + 1} / {EPOCHS} ---")
    
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    print(f"Training State Loss metrics: {train_loss:.4f} | Training State Accuracy rates: {train_acc * 100:.2f}%")
    
    val_loss, val_acc, val_preds, val_labels = eval_model(model, test_loader, criterion, device)
    print(f"Validation State Loss metrics: {val_loss:.4f} | Validation State Accuracy rates: {val_acc * 100:.2f}%")

print("\nOptimization processing loop completed.")

Initiating sequential RNN training processing loop execution...

--- Epoch 1 / 5 ---
Training State Loss metrics: 0.5950 | Training State Accuracy rates: 67.12%
Validation State Loss metrics: 0.5231 | Validation State Accuracy rates: 73.57%

--- Epoch 2 / 5 ---
Training State Loss metrics: 0.4740 | Training State Accuracy rates: 78.33%
Validation State Loss metrics: 0.3946 | Validation State Accuracy rates: 82.22%

--- Epoch 3 / 5 ---
Training State Loss metrics: 0.4786 | Training State Accuracy rates: 77.44%
Validation State Loss metrics: 0.5994 | Validation State Accuracy rates: 68.97%

--- Epoch 4 / 5 ---
Training State Loss metrics: 0.4763 | Training State Accuracy rates: 77.26%
Validation State Loss metrics: 0.3986 | Validation State Accuracy rates: 83.28%

--- Epoch 5 / 5 ---
Training State Loss metrics: 0.4480 | Training State Accuracy rates: 80.67%
Validation State Loss metrics: 0.5274 | Validation State Accuracy rates: 73.91%

Optimization processing loop completed.


Adam optimizer: a popular optimizer, which is a strong default for RNNs and most NNs As it doesn't take much tuning and adapts the learning rate per parameter automatically.


With five epochs, it looks like we converged early before the loss metric spiked and the accuracy plateaued.


In [74]:
print("Final Evaluation Assessment Classification Report:\n")
print(classification_report(val_labels, val_preds, target_names=['0 (Neutral)', '1 (Biased)']))

Final Evaluation Assessment Classification Report:

              precision    recall  f1-score   support

 0 (Neutral)       0.72      0.77      0.74      1473
  1 (Biased)       0.76      0.71      0.73      1524

    accuracy                           0.74      2997
   macro avg       0.74      0.74      0.74      2997
weighted avg       0.74      0.74      0.74      2997



In [76]:
# Simulation/test

test_sentence = "The corporate tax rate was adjusted by two percent"

# Convert test text sentence into clean numerical token streams
tokens = clean_and_tokenize(test_sentence)
numerical_sequence = [vocab.get(token, vocab['<UNK>']) for token in tokens]

# Apply sequence truncation or padding rules to match length constraints
if len(numerical_sequence) > 64:
    numerical_sequence = numerical_sequence[:64]
else:
    numerical_sequence = numerical_sequence + [vocab['<PAD>']] * (64 - len(numerical_sequence))

# Wrap sequence matrix inside tracking tensor array blocks
input_tensor = torch.tensor([numerical_sequence], dtype=torch.long).to(device)

model.eval()
with torch.no_grad():
    raw_logits = model(input_tensor)
    
    # Softmax 
    probabilities = torch.softmax(raw_logits, dim=1).flatten()
    prediction = torch.argmax(raw_logits, dim=1).item()

print(f"User Input String: '{test_sentence}'\n")
if prediction == 1:
    print(f"Final Model Verdict: 🚨 BIASED Language Detected")
    print(f"Model Classification Probability Rate: {probabilities[1].item() * 100:.2f}%")
else:
    print(f"Final Model Verdict: ✅ NEUTRAL Language Detected")
    print(f"Model Classification Probability Rate: {probabilities[0].item() * 100:.2f}%")

User Input String: 'The corporate tax rate was adjusted by two percent'

Final Model Verdict: 🚨 BIASED Language Detected
Model Classification Probability Rate: 67.05%
